# Multi-Agent Session Persistence

Session managers also work with multi-agent patterns. Both `Swarm` and `Graph`
accept a `session_manager` parameter that persists the state of all participating agents
and the orchestrator itself.

This notebook demonstrates session persistence with:
1. A **Swarm** of collaborating agents
2. A **Graph** of agents in a deterministic workflow

In [ ]:
%pip install -q --upgrade strands-agents

## Part 1: Swarm with Session Persistence

A Swarm lets agents hand off tasks to each other. With a session manager,
the full conversation history, handoff state, and shared context persist
to the configured storage backend.

In [ ]:
from strands import Agent
from strands.multiagent.swarm import Swarm
from strands.session.file_session_manager import FileSessionManager

SESSION_ID = "swarm-session-demo"
STORAGE_DIR = "./sessions"

# Create specialized agents
researcher = Agent(
    system_prompt="You are a research assistant. You find and summarize information on topics.",
    name="researcher",
)

writer = Agent(
    system_prompt="You are a technical writer. You take research summaries and produce clear documentation.",
    name="writer",
)

# Create a session manager for the swarm
session_manager = FileSessionManager(
    session_id=SESSION_ID,
    storage_dir=STORAGE_DIR,
)

swarm = Swarm(
    nodes=[researcher, writer],
    entry_point=researcher,
    session_manager=session_manager,
)

result = swarm(
    "Research the key benefits of event-driven architecture for microservices."
)
print(result)

### Verify the Swarm session was persisted

The session manager saves each agent's conversation history and the swarm's
orchestration state. Let's inspect what was written to disk.

In [ ]:
import os

swarm_session_dir = os.path.join(STORAGE_DIR, f"session_{SESSION_ID}")
for root, dirs, files in os.walk(swarm_session_dir):
    level = root.replace(swarm_session_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for file in files:
        print(f"{sub_indent}{file}")

## Part 2: Graph with Session Persistence

A Graph defines a deterministic workflow where agents execute as nodes.
Session persistence saves the state of each node agent.

In [ ]:
from strands import Agent
from strands.multiagent.graph import GraphBuilder
from strands.session.file_session_manager import FileSessionManager

GRAPH_SESSION_ID = "graph-session-demo"

# Define agents for the graph
planner = Agent(
    system_prompt="You are a project planner. Break down tasks into actionable steps.",
    name="planner",
)

executor = Agent(
    system_prompt="You are a task executor. Take a plan and describe how you would implement each step.",
    name="executor",
)

reviewer = Agent(
    system_prompt="You are a code reviewer. Review implementation plans for completeness and correctness.",
    name="reviewer",
)

# Build the graph: planner -> executor -> reviewer
graph_session_manager = FileSessionManager(
    session_id=GRAPH_SESSION_ID,
    storage_dir=STORAGE_DIR,
)

builder = GraphBuilder()
planner_node = builder.add_node(planner)
executor_node = builder.add_node(executor)
reviewer_node = builder.add_node(reviewer)
builder.add_edge(planner_node, executor_node)
builder.add_edge(executor_node, reviewer_node)
builder.set_session_manager(graph_session_manager)
graph = builder.build()

result = graph(
    "Plan and review the implementation of a REST API for a todo list application."
)
print(result)

## Inspect the multi-agent session structure

Multi-agent sessions store state for each participating agent.

In [ ]:
import os

for root, dirs, files in os.walk(STORAGE_DIR):
    level = root.replace(STORAGE_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for file in files:
        print(f"{sub_indent}{file}")

## Cleanup

In [ ]:
import shutil

shutil.rmtree(STORAGE_DIR, ignore_errors=True)
print("Session files cleaned up.")